# Dispatch baseline test (rule-based, no optimization)

In [ ]:
%load_ext autoreload
%autoreload 2

from bess_office.fetch_pv import get_pv_profile
from bess_office.fetch_load import get_load_profile
from bess_office.dispatch_sim import run_baseline_dispatch, compute_dispatch_metrics, BatterySpec

## Fetch and align data

In [ ]:
pv = get_pv_profile(latitude=52.01, longitude=4.36, peak_power_kw=10)
load = get_load_profile(peak_kwh_per_quarter=15)
load_hourly = load.resample("1h").sum()  # match PV's hourly resolution

## Run the baseline dispatch

In [ ]:
peak_threshold_kw = 8

battery = BatterySpec(capacity_kwh=10, power_kw=10)
result = run_baseline_dispatch(
    pv, load_hourly, battery,
    peak_threshold_kw=peak_threshold_kw,
    timestep_hours=1.0,
)
result.head()

## Peak shaving result

In [ ]:
metrics = compute_dispatch_metrics(result, peak_threshold_kw=peak_threshold_kw)

print(f"Peak without battery: {metrics['peak_import_no_battery_kw']:.2f} kW")
print(f"Peak with battery:    {metrics['peak_import_with_battery_kw']:.2f} kW")
print(f"Peak shaved:          {metrics['peak_shaved_kw']:.2f} kW ({metrics['peak_shaved_pct']:.0f}%)")
print()
print(f"Self-consumption (no battery):   {metrics['self_consumption_no_battery_pct']:.0f}%")
print(f"Self-consumption (with battery): {metrics['self_consumption_pct']:.0f}%")
print(f"Self-consumption gain:           +{metrics['self_consumption_gain_pct']:.0f} pp")
print()
print(f"Total grid import: {result['grid_import_kwh'].sum():.2f} kWh")
print(f"Total grid export: {result['grid_export_kwh'].sum():.2f} kWh")